In [0]:
%pip install --upgrade databricks-sdk PyJWT
dbutils.library.restartPython()

## iOS Authentication Dance — End-to-End Simulation

This notebook simulates the full iOS auth flow on the Databricks side:

1. **SPN OAuth Token** — iOS app uses the bootstrap SPN credentials to get a short-lived Databricks OAuth token (`client_credentials` grant)
2. **Auth Exchange** — iOS app calls `POST /api/v1/auth/apple/exchange` with the SPN bearer token + Apple identity token. Server validates Apple token, creates user in Lakebase, returns app JWT
3. **JWT-Authenticated Ingest** — iOS app sends `Authorization: Bearer <SPN token>` (for sidecar auth) + `X-User-JWT: <app JWT>` (for user identity) to `POST /api/v1/healthkit/ingest`

**Auth model:** The SPN OAuth token authenticates the device to Databricks (sidecar). The app JWT is a signed identity envelope that tells the server *who* the user is. Both travel on every ingest request.

> **Note:** Step 2 requires a real Apple identity token (signed by Apple's JWKS keys), which we can't produce from a notebook. The exchange call will reach the server (proving SPN auth + route guard work) but fail at Apple JWKS validation. We then **craft the JWT ourselves** using the signing secret to simulate step 3.

In [0]:
from databricks.sdk import WorkspaceClient
import json

wc = WorkspaceClient()
WORKSPACE_HOST = wc.config.host.rstrip('/')

# ── iOS Bootstrap SPN credentials (from infra bundle's secret scope) ──
SCOPE = 'dbxw_zerobus_credentials'
SPN_CLIENT_ID = dbutils.secrets.get(SCOPE, 'ios_spn_application_id')
SPN_CLIENT_SECRET = dbutils.secrets.get(SCOPE, 'ios_bootstrap_client_secret_dev_matthew_giglia_wearables')

# ── JWT signing secret (same key the server uses to sign/verify app JWTs) ──
JWT_SIGNING_SECRET = dbutils.secrets.get(SCOPE, 'jwt_signing_secret')

# ── Apple Bundle ID (must match iOS PRODUCT_BUNDLE_IDENTIFIER) ────────
# This is the `aud` claim Apple embeds in identity tokens. The server
# validates: aud === APPLE_BUNDLE_ID. If these don't match, the exchange
# endpoint returns AUDIENCE_MISMATCH.
APPLE_BUNDLE_ID = dbutils.secrets.get(SCOPE, 'apple_bundle_id')

# ── App URL (discovered from SDK, not hardcoded) ──
APP_URL = None
for app in wc.apps.list():
    if app.name == 'dev-dbxw-0bus-ingest':
        APP_URL = app.url.rstrip('/')
        break
assert APP_URL, 'App not found!'

print(f'Workspace:       {WORKSPACE_HOST}')
print(f'App URL:         {APP_URL}')
print(f'SPN Client:      {SPN_CLIENT_ID[:8]}...')
print(f'JWT Secret:      {JWT_SIGNING_SECRET[:6]}... ({len(JWT_SIGNING_SECRET)} chars)')
print(f'Apple Bundle ID: {APPLE_BUNDLE_ID}')
print(f'\n✓ All credentials loaded')

In [0]:
import requests

# ── Client credentials grant (what the iOS AuthService.swift does) ────
# The iOS app stores the SPN client_id and client_secret in the Keychain
# (provisioned via QR code scan). It calls the workspace OIDC endpoint
# to get a short-lived access token before any API calls.

token_url = f'{WORKSPACE_HOST}/oidc/v1/token'
resp = requests.post(token_url, data={
    'grant_type': 'client_credentials',
    'client_id': SPN_CLIENT_ID,
    'client_secret': SPN_CLIENT_SECRET,
    'scope': 'all-apis',
}, timeout=15)

assert resp.status_code == 200, f'OAuth failed: {resp.status_code} {resp.text}'
oauth = resp.json()
SPN_TOKEN = oauth['access_token']

print(f'✓ SPN OAuth token acquired')
print(f'  Token type:  {oauth["token_type"]}')
print(f'  Expires in:  {oauth["expires_in"]}s')
print(f'  Token:       {SPN_TOKEN[:20]}...')

In [0]:
import hashlib, secrets as py_secrets, base64

# ── Simulate the iOS Sign in with Apple request ────────────────────
# In production, the iOS app would:
#   1. Trigger ASAuthorizationAppleIDProvider
#   2. User authenticates with Face ID / password
#   3. Apple returns an identity_token JWT (signed with Apple's RS256 keys)
#   4. iOS sends that token to our server
#
# We construct a structurally valid but unsigned Apple identity token.
# Expected result: route guard allows ios-spn → auth zone,
# but Apple JWKS validation fails → 401 APPLE_AUTH_FAILED

raw_nonce = py_secrets.token_hex(16)
nonce_hash = hashlib.sha256(raw_nonce.encode()).hexdigest()

# ── Build fake Apple identity token with correct structure ───────────
# The `aud` claim MUST match APPLE_BUNDLE_ID from the secret scope
# (i.e., the iOS app's PRODUCT_BUNDLE_IDENTIFIER). Apple embeds this
# automatically when the user authenticates through the app.

def _b64url(data: dict) -> str:
    """Base64url-encode a JSON dict (no padding), matching JWT spec."""
    return base64.urlsafe_b64encode(json.dumps(data, separators=(',', ':')).encode()).rstrip(b'=').decode()

fake_apple_header = _b64url({'kid': 'W6WcOKB', 'alg': 'RS256', 'typ': 'JWT'})
fake_apple_payload = _b64url({
    'iss': 'https://appleid.apple.com',
    'sub': 'fake-apple-sub-mickey',
    'aud': APPLE_BUNDLE_ID,          # ← from secret scope (com.dbxwearables.dbxWearablesApp)
    'exp': 9999999999,
    'iat': 1700000000,
    'nonce': nonce_hash,
    'email': 'mickey.mouse@databricks.com',
    'email_verified': True,
    'is_private_email': False,
    'auth_time': 1700000000,
})
fake_apple_token = f'{fake_apple_header}.{fake_apple_payload}.fake-signature-not-signed-by-apple'

exchange_url = f'{APP_URL}/api/v1/auth/apple/exchange'
exchange_payload = {
    'appleIdToken': fake_apple_token,
    'nonce': raw_nonce,
    'userId': 'fake-apple-sub-mickey',
    'deviceId': 'test-device-mickey-001',
}

print(f'POST {exchange_url}')
print(f'Authorization: Bearer <SPN token>')
print(f'\nFake Apple token claims:')
print(f'  iss: https://appleid.apple.com')
print(f'  sub: fake-apple-sub-mickey')
print(f'  aud: {APPLE_BUNDLE_ID}')
print(f'  nonce: {nonce_hash[:16]}...')
print(f'\nExchange body:')
print(json.dumps(exchange_payload, indent=2))
print()

resp = requests.post(
    exchange_url,
    json=exchange_payload,
    headers={'Authorization': f'Bearer {SPN_TOKEN}', 'Content-Type': 'application/json'},
    timeout=15,
    allow_redirects=True,
)

print(f'Status: {resp.status_code}')
try:
    body = resp.json()
    print(json.dumps(body, indent=2))
except Exception:
    print(resp.text[:1000])

# ── Interpret the result ──────────────────────────────────────────────────────
if resp.status_code == 401:
    code = body.get('code', '')
    if code in ('APPLE_AUTH_FAILED', 'SIGNATURE_INVALID', 'TOKEN_EXPIRED'):
        print(f'\n✓ Route guard allowed ios-spn to auth zone')
        print(f'  Apple JWKS validation failed as expected (no real Apple signature)')
        print(f'  Error code: {code}')
    else:
        print(f'\n⚠ Got 401 but unexpected code: {code}')
        print(f'  This may be the sidecar rejecting the SPN token')
elif resp.status_code == 200:
    print(f'\n✓ Exchange succeeded! (unexpected with a fake token)')
else:
    print(f'\n⚠ Unexpected status: {resp.status_code}')

In [0]:
import jwt as pyjwt
import uuid
import time

# ── Simulate what the server does after successful Apple validation ───
#
# In production, auth-service.ts would:
#   1. Validate the Apple identity token via JWKS
#   2. Extract sub claim (Apple's stable user ID)
#   3. Upsert user in Lakebase: auth.users (apple_sub → user_id UUID)
#   4. Register device in Lakebase: auth.devices
#   5. Issue HS256 JWT with { sub: user_id, device_id, platform }
#
# Since we can't complete Apple validation from a notebook, we craft
# the JWT directly using the same signing secret the server uses.

MICKEY_USER_ID = str(uuid.uuid4())
DEVICE_ID = 'test-device-mickey-001'
PLATFORM = 'apple_healthkit'

payload = {
    'sub': MICKEY_USER_ID,
    'device_id': DEVICE_ID,
    'platform': PLATFORM,
    'iat': int(time.time()),
    'exp': int(time.time()) + 900,  # 15 minutes (matches ACCESS_TOKEN_EXPIRY)
}

APP_JWT = pyjwt.encode(payload, JWT_SIGNING_SECRET, algorithm='HS256')

print('✓ App JWT crafted for Mickey Mouse')
print(f'  user_id (sub): {MICKEY_USER_ID}')
print(f'  device_id:     {DEVICE_ID}')
print(f'  platform:      {PLATFORM}')
print(f'  expires:       {time.strftime("%H:%M:%S", time.gmtime(payload["exp"]))} UTC')
print(f'  JWT:           {APP_JWT[:40]}...')
print()

# Decode to verify it's valid
decoded = pyjwt.decode(APP_JWT, JWT_SIGNING_SECRET, algorithms=['HS256'])
print(f'  Decoded claims: {json.dumps(decoded, indent=2)}')

In [0]:
import random
from datetime import datetime, timedelta, timezone

# ── Generate a random workout ─────────────────────────────────────────
now = datetime.now(timezone.utc)
workout_start = now - timedelta(minutes=random.randint(30, 120))
workout_duration = random.randint(1200, 3600)  # 20–60 min
workout_end = workout_start + timedelta(seconds=workout_duration)

workout = {
    'workoutActivityType': 'HKWorkoutActivityTypeRunning',
    'startDate': workout_start.isoformat(),
    'endDate': workout_end.isoformat(),
    'duration': float(workout_duration),
    'totalDistance': {
        'value': round(random.uniform(2.0, 10.0), 3),
        'unit': 'km',
    },
    'totalEnergyBurned': {
        'value': round(random.uniform(200.0, 800.0), 1),
        'unit': 'kcal',
    },
    'source': {
        'name': 'Apple Watch',
        'bundleIdentifier': 'com.apple.health',
    },
    'metadata': {
        'HKIndoorWorkout': False,
        'HKWeatherTemperature': f'{random.randint(55, 85)} degF',
        'testUser': 'Mickey Mouse',
        'testEmail': 'mickey.mouse@databricks.com',
    },
}

ndjson_body = json.dumps(workout)

print('Generated workout:')
print(json.dumps(workout, indent=2))
print(f'\nNDJSON payload ({len(ndjson_body)} bytes)')

# ── POST to the ingest endpoint ─────────────────────────────────────
#
# New auth model:
#   Authorization: Bearer <SPN OAuth token>  → authenticates with sidecar
#   X-User-JWT: <app JWT>                    → identifies the user to the server
#
# The sidecar validates the SPN token and forwards the request.
# The route guard classifies the caller as 'ios-spn' (allowed in ingest zone).
# optionalAuth middleware reads X-User-JWT, validates it, populates req.auth.
# extractUser() reads req.auth.sub → Mickey's user_id lands in the bronze table.

ingest_url = f'{APP_URL}/api/v1/healthkit/ingest'

print(f'\nPOST {ingest_url}')
print(f'  Authorization: Bearer <SPN token>')
print(f'  X-User-JWT:    {APP_JWT[:30]}...')
print(f'  X-Record-Type: workouts')
print()

resp = requests.post(
    ingest_url,
    data=ndjson_body,
    headers={
        'Authorization': f'Bearer {SPN_TOKEN}',
        'X-User-JWT': APP_JWT,
        'Content-Type': 'application/x-ndjson',
        'X-Record-Type': 'workouts',
        'X-Device-Id': DEVICE_ID,
        'X-Platform': PLATFORM,
        'X-App-Version': '1.0.0-test',
    },
    timeout=15,
    allow_redirects=True,
)

print(f'Status: {resp.status_code}')
try:
    body = resp.json()
    print(json.dumps(body, indent=2))
except Exception:
    print(resp.text[:1000])

if resp.status_code == 200:
    print(f'\n✓ Mickey Mouse\'s workout ingested!')
    print(f'  user_id: {MICKEY_USER_ID}')
    print(f'  The bronze table now has a record attributed to Mickey via X-User-JWT.')
elif resp.status_code == 403:
    caller = body.get('caller_type', 'unknown')
    zone = body.get('zone', 'unknown')
    print(f'\n✗ Route guard denied: caller={caller}, zone={zone}')
    print(f'  This means the classifyRoute fix or access matrix update is not deployed yet.')
    print(f'  Run: cd dbxW_zerobus_app && databricks bundle deploy --target dev')
else:
    print(f'\n⚠ Unexpected status {resp.status_code}')

In [0]:
%sql
-- Recent ERROR-level logs from the app's OTel telemetry export

SELECT
  time,
  body::STRING AS error_message,
  attributes:`app.log_source`::STRING AS log_source,
  attributes:`app.deployment_id`::STRING AS deployment_id
FROM hls_fde_dev.dev_matthew_giglia_wearables.dbxw_0bus_ingest_otel_logs
WHERE severity_text = 'ERROR'
  AND date >= current_date() - INTERVAL 7 DAYS
ORDER BY time DESC
LIMIT 50

In [0]:
%sql
-- Poll for deployment logs after the preinstall fix.
-- Re-run this cell every ~60s to check progress.
-- Look for: "Starting deployment" → file uploads → "Installing dependencies"
--   → build logs → "App is running" (success) or ERROR (failure)

SELECT
  time,
  severity_text,
  body::STRING AS msg
FROM hls_fde_dev.dev_matthew_giglia_wearables.dbxw_0bus_ingest_otel_logs
WHERE date = current_date()
  AND time > '2026-04-28T13:50:39Z'  -- after bundle deploy with preinstall fix
ORDER BY time ASC
LIMIT 200

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import AppDeployment

wc = WorkspaceClient()
app_name = 'dev-dbxw-0bus-ingest'

# Check current state
app = wc.apps.get(app_name)
print(f'Compute: {app.compute_status.state.value if app.compute_status else "unknown"}')
if app.active_deployment:
    print(f'Active deployment: {app.active_deployment.deployment_id} ({app.active_deployment.status.state.value})')
else:
    print('No active deployment')

print(f'\nTriggering redeploy with .npmrc throttling (maxsockets=2)...')
print('This reduces memory pressure during npm install.')
print('Waiting for build (~5-10 min)...\n')

try:
    deployment = wc.apps.deploy(app_name, AppDeployment()).result()
    print(f'\u2713 Deployment succeeded!')
    print(f'  Deployment ID:  {deployment.deployment_id}')
    print(f'  Status:         {deployment.status.state.value}')
    print(f'  Message:        {deployment.status.message}')
except Exception as e:
    print(f'\u2717 Deployment failed: {e}')